[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [1]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [9]:
data_frame = pd.read_csv("Dropped_data.csv")

#### Deleting unwanted features
Some features are not needed or are less workable than others, removing them cleans up the data and provides easier, simpler ways to manipualate the data present

In [17]:
Dropped_match_data_v5 = data_frame.drop(
    columns=[
        "blueTeamControlWardsPlaced",
        "blueTeamWardsPlaced",
        "blueTeamDragonKills",
        "BlueTeamHeraldKills",
        "blueTeamInhibitorsDestroyed",
        "BlueTeamTurretPlatesDestroyed",
        "blueTeamJungleMinions",
        "blueTeamTotalDamageToChamps",
        "redTeamControlWardsPlaced",
        "redTeamWardsPlaced",
        "redTeamDragonKills",
        "redTeamHeraldKills",
        "redTeamInhibitorsDestroyed",
        "redTeamTurretPlatesDestroyed",
        "redTeamJungleMinions",
        "redTeamTotalDamageToChamps",
    ],
    errors="ignore",
)

Dropped_match_data_v5.to_csv(
    "../2.1.Data_Wrangling/Dropped_data.csv", index=False
)

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [ ]:
data_frame.isnull().sum()

If you have null data there are many ways to deal with the empty/null values. These are the two most common approaches.
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Students should reflect why this example removes the null 'SEX' but replacing the mean 'Target'?

In [ ]:
# Remove Null values
data_frame = data_frame.dropna(subset=['SEX'])
data_frame.isnull().sum()

In [ ]:
# Replace Null values with the mean value for the column
data_frame['Target'] = data_frame['Target'].fillna(data_frame['Target'].mean())
data_frame.isnull().sum()

#### Remove Duplicates

Duplicate data can have detrimental effects on your machine learning models and outcomes, such as reducing data diversity and representativeness, which can lead to overfitting or biased models.

The `duplicated().sum()` method call returns the count of duplicate rows in the data frame.

In [10]:
data_frame.duplicated().sum()

np.int64(7)

The `drop_duplicates()` method call can be then stored back onto the data_frame variable removing the duplicates.

In [11]:
data_frame = data_frame.drop_duplicates()
data_frame.duplicated().sum()

np.int64(0)

#### Replace data

We can run a lambda function on a column to modify its values. For a simple example, let’s convert the Sex to lowercase. To run a function over a complete column, we can use the apply method which iterates over each row and modifies the values.

In [ ]:
data_frame['SEX'] = data_frame['SEX'].apply(lambda x: x.lower())
data_frame['SEX'].head()

We can check that there are no data entry errors by the `unique()` method call.

In [14]:
data_frame['blueTeamTotalKills'].unique()

array([ 4, 12, 13,  8, 11,  3, 15,  7,  6,  9, 21, 25, 16, 18, 19, 22, 14,
       10, 17, 24, 28, 20,  5, 23,  2, 27, 26,  1, 30, 33, 35, 29, 32,  0,
       31, 38, 34, 37])

In [ ]:
data_frame['SEX'] = data_frame['SEX'].apply(lambda gender: 'male' if gender.lower() == 'male' else 'female')
data_frame['SEX'].unique()

#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [15]:
#get the inter-quartile range on the blood pressure column
print(data_frame['blueTeamTotalKills'].describe())
Q1 = data_frame['blueTeamTotalKills'].quantile(0.25)
Q3 = data_frame['blueTeamTotalKills'].quantile(0.75)
IQR = Q3 - Q1
print(f'Outliers are a blueTeamTotalKills above {Q3 + IQR * 1.5} or below {Q1 - IQR * 1.5}')


count    24218.000000
mean        12.790941
std          4.909179
min          0.000000
25%          9.000000
50%         12.000000
75%         16.000000
max         38.000000
Name: blueTeamTotalKills, dtype: float64
Outliers are a blueTeamTotalKills above 26.5 or below -1.5


In [16]:
# Filter  blueTeamTotalKills within the acceptable range
data_frame = data_frame[
    (data_frame["blueTeamTotalKills"] >= Q1 - 1.5 * IQR)
    & (data_frame["blueTeamTotalKills"] <= Q3 + 1.5 * IQR)
]
print(data_frame["blueTeamTotalKills"].describe())

count    24035.000000
mean        12.669856
std          4.723191
min          0.000000
25%          9.000000
50%         12.000000
75%         16.000000
max         26.000000
Name: blueTeamTotalKills, dtype: float64


#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [ ]:
scale_feature = 'BP'

#the minimum value with space for outliers
MIN_BP = 55

#the maximum value with space for outliers
MAX_BP = 140

#scale features
data_frame[scale_feature] = [(X - MIN_BP) / (MAX_BP - MIN_BP) for X in data_frame[scale_feature]]

data_frame.describe()

> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [ ]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False, headers=True)